# M0 — Baseline Language Model

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
!pip -q install -U "transformers>=4.41" "accelerate>=0.31" "bitsandbytes>=0.43" huggingface_hub hf-transfer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, time

In [ ]:
!apt-get -y update && apt-get -y install build-essential

In [ ]:
!pip install -q pandas fastparquet ninja
# since downloaded the wheels to Google drive, only need to download it from drive
# Optional ExLlamaV2 acceleration requires a Databricks-compatible wheel installed by the workspace administrator.

In [ ]:
import os
os.environ["EXLLAMAV2_VERBOSE"] = "1"

# After loading EXLLAMAV2
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, time

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

In [ ]:
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    exllama_config={"version": 2},
)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

In [ ]:
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quant,
    low_cpu_mem_usage=True,
).eval()

print(f"Loaded in {time.time()-t0:.1f}s")
print("Model class :", type(model))

In [ ]:
import pandas as pd
# Google Colab file and Drive helpers removed for Databricks compatibility.
import pathlib

# Read the dataset
file_path = pathlib.Path("/Volumes/main/default/thesis_project/GovernmentDocument/gov-report-qs/processed_20250804_234209/qs_test_qa_evidence.parquet")

In [ ]:
df = pd.read_parquet(file_path)
print(f"✅ Loaded:{file_path.name}  |  rows = {len(df)}")
print("Columns:", df.columns.tolist())
df.head(3)

In [ ]:
import torch, re
from torch.nn.utils.rnn import pad_sequence

In [ ]:
def clean_answer(answer: str, question: str) -> str:
    """
    Clean the generated answer by removing unwanted artifacts.
    """
    # Remove common prefixes that models sometimes add
    prefixes_to_remove = [
        "Answer:", "A:", "The answer is:", "Response:",
        "In no more than", "Word count:", "Length:",
        "Question:", "Q:", question.strip()
    ]

    cleaned = answer.strip()

    # Remove prefixes (case insensitive)
    for prefix in prefixes_to_remove:
        if cleaned.lower().startswith(prefix.lower()):
            cleaned = cleaned[len(prefix):].strip()
            cleaned = cleaned.lstrip(":").strip()  # Remove any remaining colons

    # Remove patterns like "In X words:" or "(X words)"
    cleaned = re.sub(r'\b(?:in\s+)?\d+\s+words?[:\.]?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\(\d+\s+words?\)', '', cleaned, flags=re.IGNORECASE)

    # Remove word count mentions
    cleaned = re.sub(r'word count[:\s]*\d+', '', cleaned, flags=re.IGNORECASE)

    # Remove repeated question at the start
    question_words = question.lower().split()[:5]  # Check first 5 words of question
    answer_words = cleaned.lower().split()

    if len(question_words) >= 3 and len(answer_words) >= len(question_words):
        if all(qw in answer_words[:len(question_words)+2] for qw in question_words[:3]):
            # Find where the question ends in the answer
            for i in range(len(answer_words)):
                if i >= len(question_words) and answer_words[i] not in question_words:
                    cleaned = ' '.join(cleaned.split()[i:])
                    break

    # Clean up extra whitespace and punctuation
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    cleaned = cleaned.lstrip('.,;:').strip()

    return cleaned

In [ ]:
def chat(
    question: str,
    system_prompt: str = "You are a helpful assistant. Provide a direct, concise answer.",
    max_new_tokens: int = 192,
    do_sample: bool = False
) -> str:
    """
    Return one clean answer for a single question.
    """
    # 1) Build prompt with system + user messages
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 2) Tokenize and move to GPU / CPU
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    # 3) Generate completion
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tok.eos_token_id,
            temperature=0.7 if do_sample else None,
            repetition_penalty=1.05,
        )

    # 4) Decode only the newly generated tokens
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_answer = tok.decode(generated, skip_special_tokens=True).strip()

    # 5) Clean the answer
    clean_answer_text = clean_answer(raw_answer, question)

    # 6) Word limit - prefer ≤120 words
    words = re.split(r"\s+", clean_answer_text)
    if len(words) > 120:
        clean_answer_text = " ".join(words[:120]).rstrip(" ,.;") + " …"

    return clean_answer_text

In [ ]:
# Improved batch helper
import torch, re, time, math
from torch.nn.utils.rnn import pad_sequence

In [ ]:
def batch_chat(
    questions: list[str],
    batch_size: int = 6,
    max_new_tokens: int = 192
) -> list[str]:
    """Fast batched inference with improved cleaning."""
    all_answers, t0 = [], time.time()
    pad_id = tok.eos_token_id

    for step, i in enumerate(range(0, len(questions), batch_size), 1):
        batch_q = questions[i:i + batch_size]

        # Build & tokenize prompts with improved system message
        system_msg = "You are a helpful assistant. Provide a direct, concise answer without repeating the question or mentioning word limits."

        encoded = [
            tok(
                tok.apply_chat_template(
                    [
                        {"role": "system", "content": system_msg},
                        {"role": "user", "content": q},
                    ],
                    tokenize=False, add_generation_prompt=True
                ),
                return_tensors="pt"
            ).input_ids.squeeze(0)
            for q in batch_q
        ]

        # LEFT-pad to one tensor
        max_len = max(e.size(0) for e in encoded)
        inputs = torch.full(
            (len(encoded), max_len), pad_id, dtype=torch.long, device=model.device
        )
        for row, e in enumerate(encoded):
            inputs[row, -e.size(0):] = e
        attention_mask = inputs.ne(pad_id)

        # Generate with improved parameters
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=pad_id,
                repetition_penalty=1.05,
                eos_token_id=tok.eos_token_id,
            )

        # Decode, clean & truncate
        for j, enc in enumerate(encoded):
            gen_ids = outputs[j][enc.shape[-1]:]
            raw_ans = tok.decode(gen_ids, skip_special_tokens=True).strip()

            # Clean the answer
            clean_ans = clean_answer(raw_ans, batch_q[j])

            # Word limit at 120 words
            words = re.split(r"\s+", clean_ans)
            if len(words) > 120:
                clean_ans = " ".join(words[:120]).rstrip(" ,.;") + " …"

            all_answers.append(clean_ans)

        # Progress tracking
        if step % 10 == 0 or step == math.ceil(len(questions) / batch_size):
            done = step * batch_size
            speed = done * 80 / (time.time() - t0)
            remaining = len(questions) - done
            eta = (remaining * 80) / speed / 60 if speed else float("inf")
            print(f"[{done}/{len(questions)}]  {speed:5.1f} tok/s | ETA ≈ {eta:5.1f} min")

    return all_answers

In [ ]:
# TEST - Generate 5 answers
test_questions = df["question"].head(5).tolist()
test_answers = batch_chat(test_questions, batch_size=5, max_new_tokens=192)

for i, (q, a) in enumerate(zip(test_questions, test_answers)):
    print(f"Q: {q}")
    print(f"A: {a}")
    print(f"Words: {len(a.split())}")
    print()

In [ ]:
from tqdm.auto import tqdm
answers = []
for i in tqdm(range(0, len(df), 12), desc="Generating"):
    batch_q = df["question"].iloc[i:i+12].tolist()
    answers.extend(batch_chat(batch_q, batch_size=12, max_new_tokens=192))

df["answer_M0"] = answers

# Save both CSV and Parquet
csv_out = "/Volumes/main/default/thesis_project/M0/Test_1.1/gov_m0_answers_1.1.csv"
parq_out = "/Volumes/main/default/thesis_project/M0/Test_1.1/gov_m0_answers_1.1.parquet"
df[["question", "answer_M0"]].to_csv(csv_out, index=False)
df[["question", "answer_M0"]].to_parquet(parq_out, index=False)
print(f"✅ Finished generating answers for government dataset")
print(f"Saved to:\n  • {csv_out}\n  • {parq_out}")

In [ ]:
# For QASPER dataset
import pandas as pd
from pathlib import Path
val_path = Path("/Volumes/main/default/thesis_project/QASPER/processed_20250805_162328/qasper_test_qa_evidence.parquet")
df_val = pd.read_parquet(val_path)
print(f"✅ loaded {len(df_val)} rows ")
print("Column Names：", df_val.columns.tolist())

In [ ]:
QUESTION_COL = "question"
MAX_TOKENS = 192
BATCH_SIZE = 12

In [ ]:
# TEST QASPER - Generate 5 answers
test_questions_qasper = df_val[QUESTION_COL].head(5).tolist()
test_answers_qasper = batch_chat(test_questions_qasper, batch_size=5, max_new_tokens=MAX_TOKENS)

In [ ]:
for i, (q, a) in enumerate(zip(test_questions_qasper, test_answers_qasper)):
    print(f"Q: {q}")
    print(f"A: {a}")
    print(f"Words: {len(a.split())}")
    print()

In [ ]:
# Uncomment below for full QASPER generation:
from tqdm.auto import tqdm
answers = []
for i in tqdm(range(0, len(df_val), BATCH_SIZE), desc="Generating"):
    batch_q = df_val[QUESTION_COL].iloc[i:i+BATCH_SIZE].tolist()
    answers.extend(batch_chat(batch_q, batch_size=BATCH_SIZE, max_new_tokens=MAX_TOKENS))

In [ ]:
df_val["answer_M0"] = answers
out_dir = Path("/Volumes/main/default/thesis_project/M0/Test_1.1")
csv_out = out_dir / "qasper_test_M0_answers.csv"
parq_out = out_dir / "qasper_test_M0_answers.parquet"
df_val[[QUESTION_COL, "answer_M0"]].to_csv(csv_out, index=False)
df_val[[QUESTION_COL, "answer_M0"]].to_parquet(parq_out, index=False)
print(f"Saved to：\n  • {csv_out}\n  • {parq_out}")